In [1]:
#!/usr/bin/env python3
"""
7_PI_RANKING_ANALYSIS_ALL_STAGES.py

Analyzes and visualizes:
1. Actual Pluralistic Ignorance (belief about others - actual own willingness)
2. LLM-predicted PI (LLM prediction - actual own willingness) FOR ALL 8 STAGES

Creates 3x3 grid figures for each model showing all stages.
"""

# ================================================================
# Fix matplotlib compatibility
# ================================================================
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr, pearsonr
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style("whitegrid")

print("="*80)
print("PLURALISTIC IGNORANCE RANKING ANALYSIS - ALL STAGES")
print("="*80)

# ================================================================
# 1. Load Data
# ================================================================

print("\n" + "="*80)
print("1. LOADING DATA")
print("="*80)

# Load predictions
df = pd.read_csv("predictions_all_stages_long.csv")
print(f"✓ Loaded predictions: {len(df)} rows")

# Load ground truth
gt_df = pd.read_csv("data_final.csv")
print(f"✓ Loaded ground truth: {len(gt_df)} countries")

# ================================================================
# 2. Calculate Actual PI
# ================================================================

print("\n" + "="*80)
print("2. CALCULATING ACTUAL PLURALISTIC IGNORANCE")
print("="*80)

# Actual PI = belief about others - actual own willingness
# Both should be in same scale (0-100)

# mean_other_willingness is already 0-1, convert to 0-100
gt_df['belief_about_others'] = gt_df['mean_other_willingness'] * 100
gt_df['actual_own_willingness'] = gt_df['mean_own_willingness'] * 100

# Calculate actual PI
gt_df['actual_PI'] = gt_df['belief_about_others'] - gt_df['actual_own_willingness']

print(f"\n✓ Calculated Actual PI for {len(gt_df)} countries")
print(f"   Range: {gt_df['actual_PI'].min():.2f} to {gt_df['actual_PI'].max():.2f}")
print(f"   Mean: {gt_df['actual_PI'].mean():.2f}")
print(f"   Median: {gt_df['actual_PI'].median():.2f}")

# Check: PI should typically be negative (people underestimate others)
negative_pi = (gt_df['actual_PI'] < 0).sum()
print(f"\n   Countries with negative PI (underestimation): {negative_pi}/{len(gt_df)} ({negative_pi/len(gt_df)*100:.1f}%)")

# ================================================================
# 3. Calculate LLM-predicted PI for all stages
# ================================================================

print("\n" + "="*80)
print("3. CALCULATING LLM-PREDICTED PI FOR ALL STAGES")
print("="*80)

# Process all stages
models = ['gpt', 'claude', 'gemini', 'llama']
stages = sorted(df['stage'].unique())

print(f"✓ Processing {len(stages)} stages")

# Create base dataframe with ground truth
analysis_base = gt_df[['countrynew', 'belief_about_others', 'actual_own_willingness', 'actual_PI']].copy()

# For each stage, calculate predicted PI
stage_data = {}

for stage in stages:
    stage_df = df[df['stage'] == stage].copy()
    
    # Get one row per country (average if multiple)
    stage_country = stage_df.groupby('countrynew').agg({
        'pred_gpt': 'mean',
        'pred_claude': 'mean',
        'pred_gemini': 'mean',
        'pred_llama': 'mean'
    }).reset_index()
    
    # Merge with ground truth
    stage_analysis = analysis_base.merge(stage_country, on='countrynew', how='inner')
    
    # Calculate predicted PI for each model
    for model in models:
        pred_col = f'pred_{model}'
        pi_col = f'predicted_PI_{model}_stage{stage}'
        
        # LLM prediction is already 0-100, own willingness is 0-100
        stage_analysis[pi_col] = stage_analysis[pred_col] - stage_analysis['actual_own_willingness']
    
    stage_data[stage] = stage_analysis
    print(f"   Stage {stage}: {len(stage_analysis)} countries")

print("\n✓ Calculated predicted PI for all stages and models")

# ================================================================
# 4. Rank Countries by Actual PI (use first stage data for consistent ranking)
# ================================================================

print("\n" + "="*80)
print("4. RANKING COUNTRIES BY ACTUAL PI")
print("="*80)

# Sort by actual PI (most negative = most underestimation = highest PI)
# Use first stage data as reference for ranking
analysis_ranked = stage_data[stages[0]].sort_values('actual_PI', ascending=True).reset_index(drop=True)
country_order = analysis_ranked['countrynew'].tolist()

print("\nTop 10 countries with MOST pluralistic ignorance (most underestimation):")
print(analysis_ranked[['countrynew', 'actual_PI', 'belief_about_others', 'actual_own_willingness']].head(10).to_string(index=False))

print("\n\nBottom 10 countries with LEAST pluralistic ignorance (least underestimation/overestimation):")
print(analysis_ranked[['countrynew', 'actual_PI', 'belief_about_others', 'actual_own_willingness']].tail(10).to_string(index=False))

# ================================================================
# 5. Calculate Correlations for all stages
# ================================================================

print("\n" + "="*80)
print("5. CORRELATION ANALYSIS - ALL STAGES")
print("="*80)

correlation_results = []

for stage in stages:
    print(f"\nStage {stage}:")
    stage_df = stage_data[stage]
    
    for model in models:
        pi_col = f'predicted_PI_{model}_stage{stage}'
        
        # Remove NaN
        mask = stage_df['actual_PI'].notna() & stage_df[pi_col].notna()
        actual = stage_df.loc[mask, 'actual_PI']
        predicted = stage_df.loc[mask, pi_col]
        
        pearson_r, pearson_p = pearsonr(actual, predicted)
        spearman_r, spearman_p = spearmanr(actual, predicted)
        
        # Calculate MAE of PI prediction
        mae_pi = np.abs(actual - predicted).mean()
        
        print(f"   {model.upper()}: r={pearson_r:.3f} (p={pearson_p:.4f}), MAE={mae_pi:.2f}pp")
        
        correlation_results.append({
            'Stage': stage,
            'Model': model.upper(),
            'Pearson r': pearson_r,
            'Pearson p': pearson_p,
            'Spearman ρ': spearman_r,
            'Spearman p': spearman_p,
            'MAE of PI': mae_pi
        })

corr_df = pd.DataFrame(correlation_results)

# ================================================================
# 6. Visualizations - 3x3 Grid for Each Model
# ================================================================

print("\n" + "="*80)
print("6. CREATING VISUALIZATIONS")
print("="*80)

# For each model, create a 3x3 grid showing all 8 stages + actual
for model in models:
    print(f"\nCreating 3x3 grid for {model.upper()}...")
    
    fig = plt.figure()
    fig.set_size_inches(24, 20)
    fig.set_dpi(300)
    matplotlib.rcParams.update({})
    
    fig.suptitle(f'{model.upper()}: Actual PI vs Predicted PI Across All Stages', 
                 fontsize=20, fontweight='bold', y=0.995)
    
    axes = []
    for i in range(3):
        row_axes = []
        for j in range(3):
            ax = fig.add_subplot(3, 3, i*3 + j + 1)
            row_axes.append(ax)
        axes.append(row_axes)
    
    # Panel positions: [0,0], [0,1], [0,2], [1,0], [1,1], [1,2], [2,0], [2,1], [2,2]
    # First panel: Actual PI
    # Next 8 panels: Stages 1-8
    
    # Panel 0,0: Actual PI (with country names)
    ax = axes[0][0]
    
    # Sort by actual PI for consistent ordering
    plot_data = stage_data[stages[0]].set_index('countrynew').loc[country_order].reset_index()
    countries = plot_data['countrynew'].values
    y_pos = np.arange(len(countries))
    
    colors_actual = ['red' if x < 0 else 'blue' for x in plot_data['actual_PI']]
    bars = ax.barh(y_pos, plot_data['actual_PI'], color=colors_actual, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
    ax.set_xlabel('PI (percentage points)', fontweight='bold', fontsize=11)
    ax.set_title('(a) Actual PI\n(Ground Truth)', fontweight='bold', fontsize=13, pad=10)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(countries, fontsize=6)  # Show country names on leftmost panel
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_xaxis()
    
    # Add mean line
    mean_actual = plot_data['actual_PI'].mean()
    ax.text(0.05, 0.98, f'Mean: {mean_actual:.1f}pp', 
            transform=ax.transAxes, verticalalignment='top', fontsize=9,
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))
    
    # Panels for each stage
    panel_labels = ['b', 'c', 'd', 'e', 'f', 'g', 'h', 'i']
    
    for idx, stage in enumerate(stages):
        row = (idx + 1) // 3
        col = (idx + 1) % 3
        ax = axes[row][col]
        
        # Get data for this stage, sorted by country_order
        stage_df = stage_data[stage].set_index('countrynew').loc[country_order].reset_index()
        pi_col = f'predicted_PI_{model}_stage{stage}'
        
        colors_pred = ['red' if x < 0 else 'blue' for x in stage_df[pi_col]]
        bars = ax.barh(y_pos, stage_df[pi_col], color=colors_pred, alpha=0.7, edgecolor='black', linewidth=0.3)
        ax.axvline(x=0, color='black', linestyle='--', linewidth=1.5, alpha=0.5)
        ax.set_xlabel('PI (percentage points)', fontweight='bold', fontsize=11)
        
        # Get correlation for this stage
        stage_corr = corr_df[(corr_df['Stage'] == stage) & (corr_df['Model'] == model.upper())]
        corr_r = stage_corr['Pearson r'].values[0]
        corr_p = stage_corr['Pearson p'].values[0]
        mae = stage_corr['MAE of PI'].values[0]
        
        sig = ""
        if corr_p < 0.001:
            sig = "***"
        elif corr_p < 0.01:
            sig = "**"
        elif corr_p < 0.05:
            sig = "*"
        
        ax.set_title(f'({panel_labels[idx]}) Stage {stage}\nr={corr_r:.2f}{sig}, MAE={mae:.1f}pp', 
                    fontweight='bold', fontsize=13, pad=10)
        ax.set_yticks(y_pos)
        ax.set_yticklabels([])  # Hide country names on stage panels
        ax.grid(True, alpha=0.3, axis='x')
        ax.invert_xaxis()
    
    plt.tight_layout()
    plt.savefig(f'pi_ranking_{model}_all_stages_3x3.png', dpi=300, bbox_inches='tight')
    plt.savefig(f'pi_ranking_{model}_all_stages_3x3.pdf', bbox_inches='tight')
    print(f"   ✓ Saved pi_ranking_{model}_all_stages_3x3.png")
    print(f"   ✓ Saved pi_ranking_{model}_all_stages_3x3.pdf")
    plt.close()

# ================================================================
# 7. Create Stage 8 Comparison Figure (All Models - Bar Chart)
# ================================================================

print("\n" + "="*80)
print("7. CREATING STAGE 8 COMPARISON FIGURE (BAR CHART)")
print("="*80)

# Create 1x5 figure: Actual + 4 models at Stage 8
fig = plt.figure()
fig.set_size_inches(28, 14)
fig.set_dpi(300)
matplotlib.rcParams.update({})

fig.suptitle('Pluralistic Ignorance: Actual vs All Models (Full Information)', 
             fontsize=20, fontweight='bold', y=0.995)

axes = []
for i in range(5):
    ax = fig.add_subplot(1, 5, i + 1)
    axes.append(ax)

# Get Stage 8 data
stage8_data = stage_data[8].set_index('countrynew').loc[country_order].reset_index()
countries = stage8_data['countrynew'].values
y_pos = np.arange(len(countries))

# Panel 0: Actual PI (with country names)
ax = axes[0]
colors_actual = ['red' if x < 0 else 'blue' for x in stage8_data['actual_PI']]
bars = ax.barh(y_pos, stage8_data['actual_PI'], color=colors_actual, alpha=0.7, edgecolor='black', linewidth=0.3)
ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.5)
ax.set_xlabel('PI (percentage points)', fontweight='bold', fontsize=13)
ax.set_title('(a) Actual PI\n(Ground Truth)', fontweight='bold', fontsize=15, pad=10)
ax.set_yticks(y_pos)
ax.set_yticklabels(countries, fontsize=7)  # Country names visible here
ax.grid(True, alpha=0.3, axis='x')
ax.invert_xaxis()
ax.set_ylim(-0.5, len(countries) - 0.5)  # Ensure consistent y-axis range

mean_actual = stage8_data['actual_PI'].mean()
ax.text(0.05, 0.98, f'Mean: {mean_actual:.1f}pp\nN={len(stage8_data)}', 
        transform=ax.transAxes, verticalalignment='top', fontsize=11,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Panels 1-4: Each model at Stage 8
panel_labels = ['b', 'c', 'd', 'e']

for idx, model in enumerate(models):
    ax = axes[idx + 1]
    
    pi_col = f'predicted_PI_{model}_stage8'
    colors_pred = ['red' if x < 0 else 'blue' for x in stage8_data[pi_col]]
    bars = ax.barh(y_pos, stage8_data[pi_col], color=colors_pred, alpha=0.7, edgecolor='black', linewidth=0.3)
    ax.axvline(x=0, color='black', linestyle='--', linewidth=2, alpha=0.5)
    ax.set_xlabel('PI (percentage points)', fontweight='bold', fontsize=13)
    
    # Get correlation for Stage 8
    stage8_corr = corr_df[(corr_df['Stage'] == 8) & (corr_df['Model'] == model.upper())]
    corr_r = stage8_corr['Pearson r'].values[0]
    corr_p = stage8_corr['Pearson p'].values[0]
    mae = stage8_corr['MAE of PI'].values[0]
    
    sig = ""
    if corr_p < 0.001:
        sig = "***"
    elif corr_p < 0.01:
        sig = "**"
    elif corr_p < 0.05:
        sig = "*"
    
    ax.set_title(f'({panel_labels[idx]}) {model.upper()}\nr={corr_r:.3f}{sig}, MAE={mae:.1f}pp', 
                fontweight='bold', fontsize=15, pad=10)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([])  # No country names on model panels
    ax.grid(True, alpha=0.3, axis='x')
    ax.invert_xaxis()
    ax.set_ylim(-0.5, len(countries) - 0.5)  # Match first panel's y-axis range

plt.tight_layout()
plt.savefig('pi_ranking_stage8_all_models_comparison.png', dpi=300, bbox_inches='tight')
plt.savefig('pi_ranking_stage8_all_models_comparison.pdf', bbox_inches='tight')
print("✓ Saved pi_ranking_stage8_all_models_comparison.png")
print("✓ Saved pi_ranking_stage8_all_models_comparison.pdf")
plt.close()

# ================================================================
# 8. Create Stage 8 Scatter Plots (2x2 Grid)
# ================================================================

print("\n" + "="*80)
print("8. CREATING STAGE 8 SCATTER PLOTS (ACTUAL PI VS PREDICTED PI)")
print("="*80)

# Create 2x2 scatter plot grid showing actual vs predicted PI for each model
fig = plt.figure()
fig.set_size_inches(14, 12)
fig.set_dpi(300)
matplotlib.rcParams.update({})

fig.suptitle('Actual PI vs LLM-Predicted PI', 
             fontsize=16, fontweight='bold', y=0.995)

# Get Stage 8 data
stage8_data = stage_data[8]

# Model positions in 2x2 grid
model_positions = {
    'gpt': (0, 0),
    'claude': (0, 1),
    'gemini': (1, 0),
    'llama': (1, 1)
}

for model, (row, col) in model_positions.items():
    ax = fig.add_subplot(2, 2, row*2 + col + 1)
    
    # Get data for this model
    pi_col = f'predicted_PI_{model}_stage8'
    
    # Remove NaN values
    mask = stage8_data['actual_PI'].notna() & stage8_data[pi_col].notna()
    actual = stage8_data.loc[mask, 'actual_PI'].values
    predicted = stage8_data.loc[mask, pi_col].values
    
    # Scatter plot
    ax.scatter(actual, predicted, alpha=0.6, s=80, edgecolors='white', linewidth=0.5)
    
    # Perfect prediction line
    min_val = min(actual.min(), predicted.min())
    max_val = max(actual.max(), predicted.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect prediction')
    
    # Calculate statistics
    pearson_r, pearson_p = pearsonr(actual, predicted)
    mae = np.abs(actual - predicted).mean()
    
    # Add statistics box
    sig = ""
    if pearson_p < 0.001:
        sig = "***"
    elif pearson_p < 0.01:
        sig = "**"
    elif pearson_p < 0.05:
        sig = "*"
    
    stats_text = f'r = {pearson_r:.3f}{sig}\nMAE = {mae:.2f}pp'
    ax.text(0.05, 0.95, stats_text, 
            transform=ax.transAxes, verticalalignment='top',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8),
            fontsize=10)
    
    # Labels and formatting
    ax.set_xlabel('Actual PI (pp)', fontsize=11, fontweight='bold')
    ax.set_ylabel('Predicted PI (pp)', fontsize=11, fontweight='bold')
    ax.set_title(model.upper(), fontsize=13, fontweight='bold')
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3)
    
    # Equal aspect ratio for better comparison
    ax.set_aspect('equal', adjustable='box')

plt.tight_layout()
plt.savefig('pi_stage8_scatter_actual_vs_predicted.png', dpi=300, bbox_inches='tight')
plt.savefig('pi_stage8_scatter_actual_vs_predicted.pdf', bbox_inches='tight')
print("✓ Saved pi_stage8_scatter_actual_vs_predicted.png")
print("✓ Saved pi_stage8_scatter_actual_vs_predicted.pdf")
plt.close()

# ================================================================
# 9. Create Summary Heatmaps
# ================================================================

print("\n" + "="*80)
print("9. CREATING SUMMARY HEATMAPS")
print("="*80)

# Heatmap 1: Correlation across stages and models
fig = plt.figure()
fig.set_size_inches(16, 6)
fig.set_dpi(300)
matplotlib.rcParams.update({})

fig.suptitle('LLM Performance in Capturing PI Patterns Across Stages', 
             fontsize=16, fontweight='bold')

# Panel A: Pearson r
ax1 = fig.add_subplot(1, 2, 1)
pivot_r = corr_df.pivot(index='Model', columns='Stage', values='Pearson r')
sns.heatmap(pivot_r, annot=True, fmt='.2f', cmap='RdYlGn', center=0, 
            cbar_kws={'label': 'Pearson r'}, ax=ax1, vmin=-0.5, vmax=1.0)
ax1.set_title('(a) Correlation by Stage', fontweight='bold', fontsize=13)
ax1.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax1.set_ylabel('Model', fontweight='bold', fontsize=11)

# Panel B: MAE
ax2 = fig.add_subplot(1, 2, 2)
pivot_mae = corr_df.pivot(index='Model', columns='Stage', values='MAE of PI')
sns.heatmap(pivot_mae, annot=True, fmt='.1f', cmap='RdYlGn_r', 
            cbar_kws={'label': 'MAE of PI (pp)'}, ax=ax2)
ax2.set_title('(b) Mean Absolute Error by Stage', fontweight='bold', fontsize=13)
ax2.set_xlabel('Stage', fontweight='bold', fontsize=11)
ax2.set_ylabel('Model', fontweight='bold', fontsize=11)

plt.tight_layout()
plt.savefig('pi_performance_heatmap_all_stages.png', dpi=300, bbox_inches='tight')
plt.savefig('pi_performance_heatmap_all_stages.pdf', bbox_inches='tight')
print("✓ Saved pi_performance_heatmap_all_stages.png")
print("✓ Saved pi_performance_heatmap_all_stages.pdf")
plt.close()

# ================================================================
# 10. Export Results
# ================================================================

print("\n" + "="*80)
print("10. EXPORTING RESULTS")
print("="*80)

# Save correlation results
corr_df.to_csv('pi_correlation_all_stages.csv', index=False)
print("✓ Saved pi_correlation_all_stages.csv")

# Save country rankings with all stages
export_data = analysis_ranked[['countrynew', 'actual_PI', 'belief_about_others', 'actual_own_willingness']].copy()

for stage in stages:
    stage_df = stage_data[stage].set_index('countrynew').loc[country_order].reset_index()
    for model in models:
        pi_col = f'predicted_PI_{model}_stage{stage}'
        export_data[pi_col] = stage_df[pi_col].values

export_data.to_csv('pi_rankings_all_stages.csv', index=False)
print("✓ Saved pi_rankings_all_stages.csv")

# ================================================================
# 11. Summary Statistics
# ================================================================

print("\n" + "="*80)
print("SUMMARY")
print("="*80)

print("\n1. ACTUAL PLURALISTIC IGNORANCE:")
print("-" * 60)
print(f"   Mean PI: {analysis_ranked['actual_PI'].mean():.2f}pp")
print(f"   Range: {analysis_ranked['actual_PI'].min():.2f} to {analysis_ranked['actual_PI'].max():.2f}pp")
print(f"   Countries with underestimation (PI < 0): {(analysis_ranked['actual_PI'] < 0).sum()}/{len(analysis_ranked)}")

print(f"\n   Most PI (most underestimation): {analysis_ranked.iloc[0]['countrynew']} ({analysis_ranked.iloc[0]['actual_PI']:.2f}pp)")
print(f"   Least PI: {analysis_ranked.iloc[-1]['countrynew']} ({analysis_ranked.iloc[-1]['actual_PI']:.2f}pp)")

print("\n\n2. AVERAGE CORRELATION BY MODEL (across all stages):")
print("-" * 60)

for model in models:
    model_corr = corr_df[corr_df['Model'] == model.upper()]
    mean_r = model_corr['Pearson r'].mean()
    mean_mae = model_corr['MAE of PI'].mean()
    print(f"   {model.upper()}: Mean r = {mean_r:.3f}, Mean MAE = {mean_mae:.2f}pp")

print("\n\n3. BEST STAGE BY MODEL:")
print("-" * 60)

for model in models:
    model_corr = corr_df[corr_df['Model'] == model.upper()]
    best_stage = model_corr.loc[model_corr['Pearson r'].idxmax()]
    print(f"   {model.upper()}: Stage {int(best_stage['Stage'])} (r = {best_stage['Pearson r']:.3f})")

print("\n\n4. CORRELATION TABLE (by stage):")
print("-" * 60)
print("\nStage | GPT    | CLAUDE | GEMINI | LLAMA")
print("-" * 50)
for stage in stages:
    row = [f"  {stage}   |"]
    for model in models:
        r = corr_df[(corr_df['Stage'] == stage) & (corr_df['Model'] == model.upper())]['Pearson r'].values[0]
        row.append(f" {r:+.3f} |")
    print("".join(row))

print("\n" + "="*80)
print("ANALYSIS COMPLETE!")
print("="*80)

print("\nFiles created:")
print("  - pi_ranking_gpt_all_stages_3x3.png / .pdf")
print("  - pi_ranking_claude_all_stages_3x3.png / .pdf")
print("  - pi_ranking_gemini_all_stages_3x3.png / .pdf")
print("  - pi_ranking_llama_all_stages_3x3.png / .pdf")
print("  - pi_ranking_stage8_all_models_comparison.png / .pdf (bar chart)")
print("  - pi_stage8_scatter_actual_vs_predicted.png / .pdf (NEW! scatter plot)")
print("  - pi_performance_heatmap_all_stages.png / .pdf")
print("  - pi_correlation_all_stages.csv")
print("  - pi_rankings_all_stages.csv")
print("\nTotal: 2 CSV files + 14 image files (7 PNG + 7 PDF)")

PLURALISTIC IGNORANCE RANKING ANALYSIS - ALL STAGES

1. LOADING DATA
✓ Loaded predictions: 1000 rows
✓ Loaded ground truth: 125 countries

2. CALCULATING ACTUAL PLURALISTIC IGNORANCE

✓ Calculated Actual PI for 125 countries
   Range: -43.96 to -2.71
   Mean: -28.70
   Median: -28.54

   Countries with negative PI (underestimation): 125/125 (100.0%)

3. CALCULATING LLM-PREDICTED PI FOR ALL STAGES
✓ Processing 8 stages
   Stage 1: 125 countries
   Stage 2: 125 countries
   Stage 3: 125 countries
   Stage 4: 125 countries
   Stage 5: 125 countries
   Stage 6: 125 countries
   Stage 7: 125 countries
   Stage 8: 125 countries

✓ Calculated predicted PI for all stages and models

4. RANKING COUNTRIES BY ACTUAL PI

Top 10 countries with MOST pluralistic ignorance (most underestimation):
        countrynew  actual_PI  belief_about_others  actual_own_willingness
            Guinea -43.958014            39.650460               83.608474
              Mali -43.817368            42.055540        

In [1]:
#!/usr/bin/env python3
"""
8_PI_SCATTER_STATISTICS.py

Calculates 95% confidence intervals and p-values for the 
Actual PI vs LLM-Predicted PI correlations (Stage 8).
"""

import pandas as pd
import numpy as np
from scipy.stats import pearsonr
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

print("="*80)
print("PI SCATTER PLOT STATISTICS: 95% CI AND P-VALUES")
print("="*80)

# ================================================================
# Load Data
# ================================================================

print("\n1. Loading data...")
df = pd.read_csv("predictions_all_stages_long.csv")
gt_df = pd.read_csv("data_final.csv")

# Calculate Actual PI
gt_df['belief_about_others'] = gt_df['mean_other_willingness'] * 100
gt_df['actual_own_willingness'] = gt_df['mean_own_willingness'] * 100
gt_df['actual_PI'] = gt_df['belief_about_others'] - gt_df['actual_own_willingness']

# Get Stage 8 predictions
stage8_df = df[df['stage'] == 8].copy()
stage8_country = stage8_df.groupby('countrynew').agg({
    'pred_gpt': 'mean',
    'pred_claude': 'mean',
    'pred_gemini': 'mean',
    'pred_llama': 'mean'
}).reset_index()

# Merge with ground truth
analysis_base = gt_df[['countrynew', 'actual_PI', 'actual_own_willingness']].copy()
stage8_data = analysis_base.merge(stage8_country, on='countrynew', how='inner')

# Calculate predicted PI
models = ['gpt', 'claude', 'gemini', 'llama']
for model in models:
    pred_col = f'pred_{model}'
    pi_col = f'predicted_PI_{model}'
    stage8_data[pi_col] = stage8_data[pred_col] - stage8_data['actual_own_willingness']

print(f"✓ Loaded data for {len(stage8_data)} countries")

# ================================================================
# Calculate Statistics with 95% CI
# ================================================================

print("\n2. Calculating statistics...")

def calculate_correlation_ci(x, y, confidence=0.95):
    """Calculate 95% CI for Pearson correlation using Fisher's z-transformation"""
    n = len(x)
    r, p_value = pearsonr(x, y)
    
    # Fisher's z-transformation
    z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    
    # Critical value
    z_crit = stats.norm.ppf((1 + confidence) / 2)
    
    # CI for z
    z_lower = z - z_crit * se
    z_upper = z + z_crit * se
    
    # Transform back to r
    r_lower = np.tanh(z_lower)
    r_upper = np.tanh(z_upper)
    
    return r, p_value, r_lower, r_upper

results_summary = []

for model in models:
    pi_col = f'predicted_PI_{model}'
    
    # Remove NaN values
    mask = stage8_data['actual_PI'].notna() & stage8_data[pi_col].notna()
    actual = stage8_data.loc[mask, 'actual_PI'].values
    predicted = stage8_data.loc[mask, pi_col].values
    
    # Calculate correlation with 95% CI
    r, p_value, r_lower, r_upper = calculate_correlation_ci(actual, predicted)
    
    # Additional statistics
    n = len(actual)
    mae = np.abs(actual - predicted).mean()
    rmse = np.sqrt(np.mean((actual - predicted)**2))
    
    # Linear regression for slope/intercept
    slope, intercept, _, _, std_err = stats.linregress(actual, predicted)
    
    results_summary.append({
        'Model': model.upper(),
        'N': n,
        'Pearson r': r,
        '95% CI Lower': r_lower,
        '95% CI Upper': r_upper,
        'p-value': p_value,
        'Slope': slope,
        'Intercept': intercept,
        'Std Error': std_err,
        'MAE (pp)': mae,
        'RMSE (pp)': rmse
    })

# ================================================================
# Create Results DataFrame and Export
# ================================================================

results_df = pd.DataFrame(results_summary)
results_df.to_csv('pi_scatter_statistics_with_ci.csv', index=False)
print("✓ Saved pi_scatter_statistics_with_ci.csv")

# ================================================================
# Print Formatted Results
# ================================================================

print("\n" + "="*80)
print("RESULTS: ACTUAL PI vs PREDICTED PI (STAGE 8)")
print("="*80)

for idx, row in results_df.iterrows():
    print(f"\n{row['Model']}:")
    print(f"  Sample size (N):        {int(row['N'])}")
    print(f"  Pearson r:              {row['Pearson r']:.4f}")
    print(f"  95% CI:                 [{row['95% CI Lower']:.4f}, {row['95% CI Upper']:.4f}]")
    
    # Format p-value with significance
    p_val = row['p-value']
    if p_val < 0.001:
        p_str = "< 0.001***"
    elif p_val < 0.01:
        p_str = f"= {p_val:.4f}**"
    elif p_val < 0.05:
        p_str = f"= {p_val:.4f}*"
    else:
        p_str = f"= {p_val:.4f}"
    
    print(f"  p-value:                {p_str}")
    print(f"  Regression slope:       {row['Slope']:.4f} (SE = {row['Std Error']:.4f})")
    print(f"  Regression intercept:   {row['Intercept']:.3f}")
    print(f"  MAE:                    {row['MAE (pp)']:.2f} percentage points")
    print(f"  RMSE:                   {row['RMSE (pp)']:.2f} percentage points")

print("\n" + "="*80)
print("SUMMARY TABLE")
print("="*80)
print("\n" + results_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

print("\n" + "="*80)
print("COMPLETE!")
print("="*80)
print("\nFile created: pi_scatter_statistics_with_ci.csv")
print("\nNote: 95% CI calculated using Fisher's z-transformation")
print("      Significance levels: * p<0.05, ** p<0.01, *** p<0.001")

PI SCATTER PLOT STATISTICS: 95% CI AND P-VALUES

1. Loading data...
✓ Loaded data for 125 countries

2. Calculating statistics...
✓ Saved pi_scatter_statistics_with_ci.csv

RESULTS: ACTUAL PI vs PREDICTED PI (STAGE 8)

GPT:
  Sample size (N):        125
  Pearson r:              0.5969
  95% CI:                 [0.4707, 0.6992]
  p-value:                < 0.001***
  Regression slope:       0.4020 (SE = 0.0487)
  Regression intercept:   -3.929
  MAE:                    13.60 percentage points
  RMSE:                   14.95 percentage points

CLAUDE:
  Sample size (N):        125
  Pearson r:              0.7736
  95% CI:                 [0.6920, 0.8357]
  p-value:                < 0.001***
  Regression slope:       0.8898 (SE = 0.0657)
  Regression intercept:   -1.750
  MAE:                    5.06 percentage points
  RMSE:                   6.51 percentage points

GEMINI:
  Sample size (N):        125
  Pearson r:              0.1466
  95% CI:                 [-0.0298, 0.3141]
  p-val

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=feb9f195-de2a-416f-b8f1-09efca4e954f' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>